In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit import ClassicalRegister
import math

sim = BasicSimulator()

## Quantum Random Number Generation

Each random bit is obtained by preparing the state $|{+}\rangle = H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ and measuring it in the computational basis. The outcome is 0 or 1 with equal probability with quantum source of randomness.

Generate bits in batches (up to 20 qubits at once) to stay within the simulator's qubit limit.

In [2]:
# Shared utility: quantum random bit generation

def quantum_random_bits(n: int) -> list[int]:
    """
    Generate n random bits by measuring n qubits each prepared in |+> = H|0>.
    Bits are generated in batches of 20 to respect the simulator qubit limit.
    This is used by ALL parties (Alice, Bob) for their random choices.
    """
    bits = []
    batch_size = 20
    while len(bits) < n:
        size = min(batch_size, n - len(bits))
        qc = QuantumCircuit(size, size)
        for i in range(size):
            qc.h(i)          # Prepare |+> = (|0> + |1>) / sqrt(2)
        qc.measure(range(size), range(size))
        t = transpile(qc, sim)
        result = sim.run(t, shots=1, memory=True).result()
        raw = result.get_memory()[0].replace(' ', '')
        # Qiskit stores qubit 0 at the rightmost position, reverse to align
        bits.extend([int(b) for b in reversed(raw)])
    return bits[:n]


# QDistribution should be approximately 50/50
test_bits = quantum_random_bits(200)
ones  = sum(test_bits)
zeros = len(test_bits) - ones
print(f"Quantum RNG check (200 bits): {zeros} zeros, {ones} ones")
print("Sample:", test_bits[:20])

Quantum RNG check (200 bits): 119 zeros, 81 ones
Sample: [0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]


## Alice: Encoding

Alice uses two bases:
- **Basis 0 (Z / Rectilinear):** $|0\rangle$ encodes bit 0, $|1\rangle$ encodes bit 1
- **Basis 1 (X / Diagonal):** $|{+}\rangle$ encodes bit 0, $|{-}\rangle$ encodes bit 1

| Bit | Basis 0 (Z) | Basis 1 (X) |
|-----|------------|-------------|
| 0   | $|0\rangle$ | $|{+}\rangle$ |
| 1   | $|1\rangle$ | $|{-}\rangle$ |

In [3]:
# ALICE

def alice_encode(bit: int, basis: int) -> QuantumCircuit:
    """
    Alice encodes a classical bit into a qubit state.

    basis=0 (Z / Rectilinear): |0> for bit=0, |1> for bit=1
    basis=1 (X / Diagonal):    |+> for bit=0, |-> for bit=1

    Returns a QuantumCircuit representing the prepared qubit
    (no measurement — the qubit is 'sent' to Bob).
    """
    qc = QuantumCircuit(1)            # Start in |0>
    if bit == 1:
        qc.x(0)                       # Flip to |1>
    if basis == 1:
        qc.h(0)                       # Rotate to diagonal basis
    return qc


# Alice's four possible states
print("Alice's encoding circuits:")
for basis in [0, 1]:
    for bit in [0, 1]:
        label = ["Z", "X"][basis]
        state = ["|0>", "|1>", "|+>", "|->"][basis * 2 + bit]
        qc = alice_encode(bit, basis)
        print(f"  bit={bit}, basis={label} → {state}")
        print(qc.draw(fold=-1))

Alice's encoding circuits:
  bit=0, basis=Z → |0>
   
q: 
   
  bit=1, basis=Z → |1>
   ┌───┐
q: ┤ X ├
   └───┘
  bit=0, basis=X → |+>
   ┌───┐
q: ┤ H ├
   └───┘
  bit=1, basis=X → |->
   ┌───┐┌───┐
q: ┤ X ├┤ H ├
   └───┘└───┘


## Bob: Measurement

Bob randomly chooses a measurement basis for each incoming qubit:
- **Basis 0 (Z):** measure directly in the computational basis
- **Basis 1 (X):** apply H first, then measure

If Bob's basis matches Alice's, he recovers Alice's bit exactly. If not, his result is random.

In [4]:
# BOB

def bob_measure(encoding_qc: QuantumCircuit, basis: int) -> int:
    """
    Bob measures the incoming qubit in his chosen basis.

    basis=0 (Z / Rectilinear): measure directly
    basis=1 (X / Diagonal):    apply H then measure

    Returns the classical bit result (0 or 1).
    """
    qc = encoding_qc.copy()           # Bob receives Alice's qubit state
    qc.add_register(ClassicalRegister(1))
    if basis == 1:
        qc.h(0)                       # Rotate back from diagonal basis
    qc.measure(0, 0)
    t = transpile(qc, sim)
    result = sim.run(t, shots=1, memory=True).result()
    return int(result.get_memory()[0].replace(' ', ''))

## Running the BB84 Protocol

Full protocol:
1. Alice and Bob independently generate random bits and bases using quantum measurements.
2. For each of the N qubits, Alice encodes and Bob measures.
3. They publicly compare bases and keep only matching-basis positions (sifting).
4. They check a random sample for errors to verify the channel is clean.

In [5]:
# PROTOCOL PARAMETERS
N = 100        # Number of qubits transmitted
THRESHOLD = 0.11  # Error rate above which an attack is suspected (11%)

print("=" * 60)
print("BB84 Protocol — No Attacker")
print("=" * 60)
print(f"Transmitting N = {N} qubits\n")

BB84 Protocol — No Attacker
Transmitting N = 100 qubits



In [6]:
# STEP 1: ALICE generates random bits and bases
print("[ALICE] Generating random bits and bases via quantum measurement...")
alice_bits  = quantum_random_bits(N)   # Secret bits Alice wants to share
alice_bases = quantum_random_bits(N)   # Alice's random choice of basis per qubit

print(f"[ALICE] Bits  (first 20): {alice_bits[:20]}")
print(f"[ALICE] Bases (first 20): {alice_bases[:20]}  (0=Z/Rectilinear, 1=X/Diagonal)")

[ALICE] Generating random bits and bases via quantum measurement...
[ALICE] Bits  (first 20): [0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1]
[ALICE] Bases (first 20): [1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0]  (0=Z/Rectilinear, 1=X/Diagonal)


In [7]:
# STEP 2: BOB generates random bases
print("[BOB] Generating random measurement bases via quantum measurement...")
bob_bases = quantum_random_bits(N)

print(f"[BOB] Bases (first 20): {bob_bases[:20]}  (0=Z/Rectilinear, 1=X/Diagonal)")

[BOB] Generating random measurement bases via quantum measurement...
[BOB] Bases (first 20): [0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1]  (0=Z/Rectilinear, 1=X/Diagonal)


In [8]:
# STEP 3: QUANTUM CHANNEL (Alice encodes, Bob measures)
print("[QUANTUM CHANNEL] Alice encodes qubits; Bob measures each one...")

bob_results = []
for i in range(N):
    # Alice encodes qubit i
    qc = alice_encode(alice_bits[i], alice_bases[i])
    # Qubit travels through channel to Bob (no eavesdropping)
    # Bob measures qubit i
    b = bob_measure(qc, bob_bases[i])
    bob_results.append(b)

print(f"[BOB] Results (first 20): {bob_results[:20]}")

[QUANTUM CHANNEL] Alice encodes qubits; Bob measures each one...
[BOB] Results (first 20): [1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1]


In [9]:
# STEP 4: SIFTING (Alice and Bob compare bases publicly)
# They announce their bases over a public (but authenticated) classical channel.
# Bits at positions where bases differ are discarded.
print("[PUBLIC CHANNEL] Alice and Bob compare bases (not bits)...")

sifted_indices = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
sifted_alice   = [alice_bits[i]  for i in sifted_indices]
sifted_bob     = [bob_results[i] for i in sifted_indices]

print(f"[SIFTING] Matching bases at {len(sifted_indices)}/{N} positions "
      f"(expected ~{N//2})")
print(f"[ALICE] Sifted key (first 20): {sifted_alice[:20]}")
print(f"[BOB]   Sifted key (first 20): {sifted_bob[:20]}")

[PUBLIC CHANNEL] Alice and Bob compare bases (not bits)...
[SIFTING] Matching bases at 59/100 positions (expected ~50)
[ALICE] Sifted key (first 20): [1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0]
[BOB]   Sifted key (first 20): [1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0]


In [10]:
# STEP 5: ERROR CHECKING
# Alice and Bob publicly compare a sample of their sifted key bits.
# These sample bits are then discarded (revealed).
# If the error rate exceeds the threshold, they suspect eavesdropping.
print("[PUBLIC CHANNEL] Comparing a sample of sifted key bits to detect errors...")

sample_size = max(1, len(sifted_alice) // 4)   # Use 25% of sifted key as sample
sample_alice = sifted_alice[:sample_size]
sample_bob   = sifted_bob[:sample_size]

errors     = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / sample_size

print(f"\n[RESULT] Sample size:   {sample_size} bits")
print(f"[RESULT] Errors found:  {errors}")
print(f"[RESULT] Error rate:    {error_rate:.2%}")
print(f"[RESULT] Threshold:     {THRESHOLD:.2%}")

if error_rate > THRESHOLD:
    print("\n[ALERT] Error rate exceeds threshold — possible eavesdropping! Aborting.")
else:
    final_key = sifted_alice[sample_size:]   # Discard sample bits
    print(f"\n[SUCCESS] Channel is clean. No attacker detected.")
    print(f"[RESULT] Final shared key length: {len(final_key)} bits")
    print(f"[RESULT] Final key (first 20):    {final_key[:20]}")
    assert sifted_alice == sifted_bob, "Keys should match without an attacker!"
    print("[VERIFY] Alice's and Bob's sifted keys are identical")

[PUBLIC CHANNEL] Comparing a sample of sifted key bits to detect errors...

[RESULT] Sample size:   14 bits
[RESULT] Errors found:  0
[RESULT] Error rate:    0.00%
[RESULT] Threshold:     11.00%

[SUCCESS] Channel is clean. No attacker detected.
[RESULT] Final shared key length: 45 bits
[RESULT] Final key (first 20):    [0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1]
[VERIFY] Alice's and Bob's sifted keys are identical


## Summary

| Step | Description |
|------|-------------|
| 1 | Alice randomly generates bits and bases using quantum measurement of $|+\rangle$ |
| 2 | Bob randomly generates bases using quantum measurement of $|+\rangle$ |
| 3 | Alice encodes each bit as a qubit in her chosen basis; Bob measures in his basis |
| 4 | Sifting: keep only positions where Alice's and Bob's bases agree |
| 5 | Error check: compare a sample; accept the key if the error rate is below the threshold |

**Without an attacker**, quantum mechanics guarantees that when Alice and Bob use the same basis, they always get the same bit. The error rate should be ~0%, well below the threshold. The remaining sifted bits form a **shared secret key** that can be used for encryption.